In [0]:
from datetime import datetime, timedelta
import pandas as pd, uuid, random

In [0]:
CATALOG = "lakeflow_ingestion"
SCHEMA = "landing_zone"

### Generate Order Changes Events linked to customers

Use random order amounts and status to simulate a real-world changes

In [0]:
def generate_cdc_orders(num_rows=20):
    operations= ["INSERT", "UPDATA", "DELETE"]

    order_data = []
    for i in range(num_rows):
        order_id = uuid.uuid4().hex[:8] # Unique order Id

        operation = random.choice(operations)
        ts = datetime.now() - timedelta(minutes=random.randint(1, 60))

        record = {
            "order_id": order_id,
            "customer_id": random.randint(1, 20),
            "order_amount": round(random.uniform(20, 500), 2),
            "order_status": random.choice(["Shipped", "Pending", "Cancelled", "Delivered"]),
            "operation":operation,
            "ts": ts.strftime("%Y-%m-%d %H:%M:%S.%f")
        }

        order_data.append((order_id, operation))

    return pd.DataFrame(order_data)

In [0]:
#Execute generate_cdc_orders() method to generate the orders data
orders_df = generate_cdc_orders()

# Convert to Spark DataFrame
df_orders = spark.createDataFrame(orders_df)

In [0]:


# Write to volumes as csv file
orders_volume_path = "/Volumes/lakeflow_ingestion/landing_zone/source/orders"
df_orders.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", "true") \
    .option("delimiter", ",") \
    .csv(orders_volume_path)